# Portfolio Risk ML Notebook

This notebook loads the IBRD data, builds the watchlist and anomaly outputs, and shows the main tables and charts used in the portfolio analysis.

In [12]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve()
candidate_roots = [repo_root, *repo_root.parents]
repo_root = next((p for p in candidate_roots if (p / 'src').exists() and (p / 'data').exists()), repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data_pipeline.silver_layer import clean_ibrd_data
df = clean_ibrd_data()
df.head()

Cleaned data saved to: /home/rigii/ATA/data/processed/ibrd_clean.csv


,Loan Number,Country / Economy,Region,Loan Type,Loan Status,Board Approval Date,Original Principal Amount (US$),Cancelled Amount (US$),Disbursed Amount (US$),Repaid to IBRD (US$),...,Last Disbursement Date,loan_age_days,loan_age_years,is_active,is_fully_repaid,is_cancelled,repayment_ratio,disbursed_percent,portfolio_status,risk_score
0,LOAN-1000,Indonesia,South Asia,Investment,Disbursing&Repaying,2024-06-18,4.462225e+08,0.0,2.659100e+08,33394081.59,...,2025-12-30,813,2.225873,True,False,False,0.125584,59.591348,Early Stage,0.874416
1,LOAN-1001,India,South Asia,Education,Disbursing&Repaying,2024-06-11,1.350896e+08,0.0,1.086504e+08,2594830.09,...,2031-07-03,820,2.245038,True,False,False,0.023882,80.428423,Early Stage,0.976118
2,LOAN-1002,India,Latin America,Investment,Effective,2004-08-01,4.203929e+08,0.0,2.825883e+08,39543019.90,...,2013-06-15,8074,22.105407,True,False,False,0.139932,67.220041,Early Stage,0.860068
3,LOAN-1003,Ghana,Europe & Central Asia,Health,Fully Repaid,2003-07-15,1.623899e+08,0.0,7.060070e+07,29192145.23,...,2012-06-19,8457,23.154004,False,True,False,0.413482,43.476035,Fully Repaid,0.586518
4,LOAN-1004,Pakistan,South Asia,Education,Repaying,2004-07-12,3.226551e+07,0.0,1.878881e+07,4893528.95,...,2013-07-13,8094,22.160164,True,False,False,0.260449,58.231862,Early Stage,0.739551


In [13]:
def calculate_risk_score(row):
    score = 0
    original_amount = row['Original Principal Amount (US$)']
    due_amount = row['Due to IBRD (US$)']
    if pd.notna(original_amount) and original_amount != 0:
        if due_amount / original_amount > 0.5:
            score += 2
    if row['repayment_ratio'] < 0.2 and row['loan_age_years'] > 10:
        score += 2
    if pd.notna(row['is_cancelled']) and row['is_cancelled']:
        score += 1
    if row['Due to IBRD (US$)'] > 100_000_000:
        score += 1
    return score

df['risk_score'] = df.apply(calculate_risk_score, axis=1)
region_summary = (
    df.groupby('Region', as_index=False)
    .agg(total_commitments=('Original Principal Amount (US$)', 'sum'), outstanding=('Due to IBRD (US$)', 'sum'), loan_count=('Loan Number', 'count'))
    .sort_values('total_commitments', ascending=False)
)
display(region_summary.head(10))

,Region,total_commitments,outstanding,loan_count
3,Latin America,7.736377e+10,3.249264e+10,269
0,Africa,7.302898e+10,3.210225e+10,248
1,East Asia,7.262716e+10,3.303893e+10,248
4,South Asia,6.772273e+10,2.767130e+10,223
2,Europe & Central Asia,6.428907e+10,2.696456e+10,212


In [14]:
risk_by_region = df.groupby('Region', as_index=False)['risk_score'].mean().sort_values('risk_score', ascending=False)
fig = px.bar(risk_by_region, x='Region', y='risk_score', title='Average Risk by Region')
fig.show()

age_bins = pd.cut(df['loan_age_years'], bins=[0, 5, 10, 20, 30, 50, 100], labels=['0-5', '5-10', '10-20', '20-30', '30-50', '50+'])
age_risk = df.assign(age_bucket=age_bins).groupby('age_bucket', as_index=False)['risk_score'].mean()
fig2 = px.line(age_risk, x='age_bucket', y='risk_score', markers=True, title='Risk by Loan Age')
fig2.show()

/tmp/ipykernel_1187461/1890071610.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_risk = df.assign(age_bucket=age_bins).groupby('age_bucket', as_index=False)['risk_score'].mean()


In [ ]:
from src.models.ml_risk import detect_anomalies, train_default_watchlist_model

watchlist_result = train_default_watchlist_model(df, threshold=0.7)
watchlist = watchlist_result['watchlist']
display(watchlist.head(10))
print(watchlist_result['metrics'])

anomalies = detect_anomalies(df, n_outliers=10)
display(anomalies[['Loan Number', 'Loan Status', 'anomaly_score', 'Due to IBRD (US$)']].head(10))

Model saved
